In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, row_number, collect_list, struct
from pyspark.sql.window import Window
from pyspark.ml.feature import Normalizer, BucketedRandomProjectionLSH

# =====================================================================
# 1. THÔNG TIN XÁC THỰC R2 (Nhớ điền Key của bạn vào đây)
# =====================================================================
R2_ACCESS_KEY = "352e66be1b3cb7fad1866c153222faf2"
R2_SECRET_KEY = "6b97ebc8310262f21fcb5e1d315ce7b3224f73a39f7813ebb38c21fa4ab30ea2"
R2_ACCOUNT_ID = "02422b2aa3d6c7acdf824dfd9259afb2"
# =====================================================================
# 2. KHỞI TẠO SPARK (Đã thêm lệnh tắt Vectorized Reader để chống OOM)
# =====================================================================
spark = SparkSession.builder \
    .appName("Step3_Recommendation_LSH") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.driver.maxResultSize", "4g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.local.dir", "/kaggle/working/spark-temp") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.endpoint", f"https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com") \
    .config("spark.hadoop.fs.s3a.access.key", R2_ACCESS_KEY) \
    .config("spark.hadoop.fs.s3a.secret.key", R2_SECRET_KEY) \
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "true") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.hadoop.fs.s3a.connection.timeout", "200000") \
    .config("spark.hadoop.fs.s3a.connection.establish.timeout", "5000") \
    .config("spark.hadoop.fs.s3a.threads.keepalivetime", "60") \
    .config("spark.hadoop.fs.s3a.paging.maximum", "5000") \
    .config("spark.hadoop.fs.s3a.attempts.maximum", "20") \
    .config("spark.hadoop.fs.s3a.retry.limit", "20") \
    .config("spark.hadoop.fs.s3a.max.total.tasks", "1000") \
    .config("spark.hadoop.fs.s3a.multipart.size", "104857600") \
    .config("spark.hadoop.fs.s3a.multipart.purge", "false") \
    .config("spark.hadoop.fs.s3a.multipart.purge.age", "86400") \
    \
    .config("spark.sql.parquet.enableVectorizedReader", "false") \
    .config("spark.sql.parquet.columnarReaderBatchSize", "1024") \
    .getOrCreate()
# =====================================================================
# 3. ĐỊNH NGHĨA ĐƯỜNG DẪN R2
# =====================================================================
BUCKET_NAME = "amazon-reviews-test"
CATEGORY = "Tools_and_Home_Improvement"

# Đầu vào
user_profile_path = f"s3a://{BUCKET_NAME}/data-preprocess/user_profiles_sample_20/{CATEGORY}/"
tfidf_path = f"s3a://{BUCKET_NAME}/data-preprocess/meta/{CATEGORY}/tfidf_vectors.parquet"

# Đầu ra
output_recommendations = f"s3a://{BUCKET_NAME}/data-preprocess/top10_recommendations/{CATEGORY}/"

# =====================================================================
# 4. ĐỌC DỮ LIỆU
# =====================================================================
print("Đang đọc User Profiles và Item Vectors...")
df_users = spark.read.parquet(user_profile_path)
df_items = spark.read.parquet(tfidf_path).select("parent_asin", col("features").alias("tfidf_features"))

# =====================================================================
# 5. CHUẨN HÓA VECTOR (Đồng bộ tên cột thành norm_features)
# =====================================================================
normalizer_user = Normalizer(p=2.0, inputCol="user_profile_vector", outputCol="norm_features")
df_users_norm = normalizer_user.transform(df_users)

normalizer_item = Normalizer(p=2.0, inputCol="tfidf_features", outputCol="norm_features")
df_items_norm = normalizer_item.transform(df_items)

# =====================================================================
# 6. THUẬT TOÁN LSH (Siết chặt cấu hình chống tràn RAM)
# =====================================================================
print("Đang huấn luyện mô hình băm LSH...")
brp = BucketedRandomProjectionLSH(
    inputCol="norm_features", 
    outputCol="hashes", 
    bucketLength=1.0,  # Xô hẹp hơn để lọc kỹ đồ vớ vẩn ngay từ đầu
    numHashTables=1    # Giảm tải cho Kaggle
)
lsh_model = brp.fit(df_items_norm)



# =====================================================================
# 7. TÌM KIẾM CẶP TƯƠNG ĐỒNG (Similarity Join)
# =====================================================================
print("Đang ghép cặp User và Item (Similarity Join)...")
# Siết ngưỡng threshold=1.0 để dọn sạch 99% lượng file rác
df_joined = lsh_model.approxSimilarityJoin(
    df_users_norm, df_items_norm, threshold=1.0, distCol="EuclideanDistance"
)

# Rút gọn bảng
df_scored = df_joined.select(
    col("datasetA.user_id").alias("user_id"),
    col("datasetB.parent_asin").alias("recommended_item"),
    col("EuclideanDistance") 
)

# =====================================================================
# 8. LỌC LẤY TOP 10 SẢN PHẨM CHO MỖI USER
# =====================================================================
print("Đang xếp hạng và lấy Top 10...")
windowSpec = Window.partitionBy("user_id").orderBy("EuclideanDistance")

df_top_k = df_scored.withColumn("rank", row_number().over(windowSpec)) \
    .filter(col("rank") <= 10)

# Gom vào 1 mảng JSON cho đẹp file đầu ra
df_final = df_top_k.groupBy("user_id").agg(
    collect_list(
        struct(
            col("recommended_item").alias("item"), 
            col("EuclideanDistance").alias("distance")
        )
    ).alias("top_10_recommendations")
)

# =====================================================================
# 9. GHI KẾT QUẢ RA R2
# =====================================================================
print(f"Đang ghi danh sách Top 10 Gợi ý lên R2 tại:\n{output_recommendations}")
df_final.write.mode("overwrite").parquet(output_recommendations)

print("XUẤT SẮC! Đồ án của bạn đã hoàn thiện luồng Content-Based Recommendation!")

# Dọn dẹp RAM
df_users_norm.unpersist()
df_items_norm.unpersist()
spark.stop()